In [6]:
import numpy as np

N = 1024

matrix = np.random.rand(N, N).astype(np.float32)
vector = np.random.rand(N).astype(np.float32)

print("Matrix Shape:", matrix.shape)
print("Vector Shape:", vector.shape)

Matrix Shape: (1024, 1024)
Vector Shape: (1024,)


In [7]:
from numba import cuda

print("CUDA Available:", cuda.is_available())

CUDA Available: True


In [8]:
!nvidia-smi

Mon Jun  1 05:59:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             29W /   70W |     107MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
from numba import cuda
import numpy as np

# Test GPU allocation
test = np.array([1, 2, 3], dtype=np.float32)

d_test = cuda.to_device(test)

print("GPU allocation successful")

GPU allocation successful


In [10]:
# Copy matrix to GPU
d_matrix = cuda.to_device(matrix)

# Copy vector to GPU
d_vector = cuda.to_device(vector)

# Create output vector on GPU
d_output = cuda.device_array(N, dtype=np.float32)

print("Matrix and Vector transferred to GPU")
print("Output array created on GPU")

Matrix and Vector transferred to GPU
Output array created on GPU


In [12]:
@cuda.jit
def matrix_vector_multiply(matrix, vector, output):
    row = cuda.grid(1)

    if row < matrix.shape[0]:
        temp = 0.0

        for j in range(matrix.shape[1]):
            temp += matrix[row, j] * vector[j]

        output[row] = temp

Matrix = 1024 × 1024
Vector = 1024 × 1
Each thread computes one row of the output.
Thread 0 → computes output[0]
Thread 1 → computes output[1]
Thread 2 → computes output[2]
...
Thread 1023 → computes output[1023]
output[row] = Σ(matrix[row][j] × vector[j])

In [13]:
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

matrix_vector_multiply[blocks_per_grid, threads_per_block](
    d_matrix,
    d_vector,
    d_output
)

cuda.synchronize()

print("Matrix-Vector Multiplication Completed")

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


Matrix-Vector Multiplication Completed


In [14]:
gpu_result = d_output.copy_to_host()

print("Output Shape:", gpu_result.shape)
print("First 5 values:")
print(gpu_result[:5])

Output Shape: (1024,)
First 5 values:
[258.1068  253.87344 259.2501  259.43845 260.28113]


In [15]:
cpu_result = np.dot(matrix, vector)

print("First 5 CPU values:")
print(cpu_result[:5])

print("\nResults Match:",
      np.allclose(cpu_result, gpu_result))

First 5 CPU values:
[258.1068  253.87347 259.25006 259.43845 260.28113]

Results Match: True


In [16]:
import time

start = time.time()

matrix_vector_multiply[blocks_per_grid, threads_per_block](
    d_matrix,
    d_vector,
    d_output
)

cuda.synchronize()

end = time.time()

print("GPU Execution Time:", end - start, "seconds")

GPU Execution Time: 0.0010716915130615234 seconds


In [17]:
import time

start = time.time()

cpu_result = np.dot(matrix, vector)

end = time.time()

print("CPU Execution Time:", end - start, "seconds")

CPU Execution Time: 0.0008068084716796875 seconds
